In [1]:
import pandas as pd
import os

aa_dir = r"C:\Users\user\Downloads\GSE148375_clean"

# Check CPD-specific files
meta_df = pd.read_csv(os.path.join(aa_dir, "checkpoint2b_metadata_relatedness_filtered.csv"))
print("Full cohort size:", len(meta_df))
print("\nCPD column stats:")
print(meta_df['cpd'].describe())
print("\nHow many have non-null/non-zero CPD (i.e. are smokers)?")
print((meta_df['cpd'] > 0).sum(), "out of", len(meta_df))

import numpy as np
confounders_cpd = np.load(os.path.join(aa_dir, "confounders_X_cpd.npy"))
print("\nconfounders_X_cpd.npy shape:", confounders_cpd.shape)

# Check the CPD-specific stability results file mentioned in your session notes
df_cpd_check = pd.read_csv(os.path.join(aa_dir, "checkpoint9_doubleml_stability_results_cpd_corrected.csv"))
print("\nCPD SNP stability file shape:", df_cpd_check.shape)
print(df_cpd_check.columns.tolist())

Full cohort size: 3036

CPD column stats:
count    3036.000000
mean       12.722003
std        14.107207
min        -9.000000
25%         0.000000
50%         0.000000
75%        26.000000
max        60.000000
Name: cpd, dtype: float64

How many have non-null/non-zero CPD (i.e. are smokers)?
1459 out of 3036

confounders_X_cpd.npy shape: (1459, 12)

CPD SNP stability file shape: (48827, 3)
['probe_id', 'stability_fraction', 'n_significant_repeats']


In [2]:
print((meta_df['cpd'] == -9).sum(), "samples with cpd == -9")
print(meta_df[meta_df['cpd'] == -9]['smoking_status'].value_counts())

16 samples with cpd == -9
smoking_status
Non-smoker    16
Name: count, dtype: int64


In [3]:
print("Any smokers with cpd == -9 or cpd <= 0?")
print(meta_df[(meta_df['smoking_status'] == 'Smoker') & (meta_df['cpd'] <= 0)].shape[0])

Any smokers with cpd == -9 or cpd <= 0?
0


In [4]:
df_cpd_check = pd.read_csv(os.path.join(aa_dir, "checkpoint9_doubleml_stability_results_cpd_corrected.csv"))
print(df_cpd_check['stability_fraction'].describe())
print((df_cpd_check['stability_fraction'] == 0).sum(), "of", len(df_cpd_check), "have stability_fraction == 0")

count    48827.000000
mean         0.001285
std          0.030872
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: stability_fraction, dtype: float64
48532 of 48827 have stability_fraction == 0


In [5]:
import pandas as pd
import numpy as np
import re
import os
import json
import gc

aa_dir = r"C:\Users\user\Downloads\GSE148375_clean"
manifest_path = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"

def strip_address_suffix(pid):
    return re.sub(r'_\d+$', '', pid)

# 1. Reuse gene reference table (genome-build data, not cohort/phenotype specific)
genes_clean = pd.read_csv(os.path.join(aa_dir, "grch37_genes_clean.csv"))
protein_coding_genes = set(genes_clean[genes_clean["Gene type"] == "protein_coding"]["gene_name"])

# 2. Map CPD's SNP list to genes (different SNP list than smoking status - MAF-filtered within smokers)
manifest_df = pd.read_csv(manifest_path, skiprows=7, low_memory=False)
manifest_df["core_name"] = manifest_df["IlmnID"].map(strip_address_suffix)
pos_lookup = manifest_df.set_index("core_name")[["Chr", "MapInfo"]]

snp_list_cpd = pd.read_csv(os.path.join(aa_dir, "checkpoint9_doubleml_stability_results_cpd_corrected.csv"))
snp_list_cpd["core_name"] = snp_list_cpd["probe_id"].map(strip_address_suffix)
snp_list_cpd = snp_list_cpd.merge(pos_lookup, left_on="core_name", right_index=True, how="left")
snp_list_cpd = snp_list_cpd.dropna(subset=["Chr", "MapInfo"])
print("CPD SNPs with resolved positions:", len(snp_list_cpd))

snp_to_gene_cpd = {}
for chrom in snp_list_cpd["Chr"].unique():
    chrom_str = str(chrom)
    genes_this_chrom = genes_clean[genes_clean["chrom"] == chrom_str].sort_values("start")
    snps_this_chrom = snp_list_cpd[snp_list_cpd["Chr"].astype(str) == chrom_str]
    if len(genes_this_chrom) == 0 or len(snps_this_chrom) == 0:
        continue
    starts = genes_this_chrom["start"].values
    ends = genes_this_chrom["end"].values
    names = genes_this_chrom["gene_name"].values
    for _, row in snps_this_chrom.iterrows():
        pos = row["MapInfo"]
        idx = np.searchsorted(starts, pos, side="right") - 1
        match = None
        for j in range(max(0, idx-3), min(len(starts), idx+4)):
            if starts[j] <= pos <= ends[j]:
                match = names[j]
                break
        snp_to_gene_cpd[row["probe_id"]] = match if match else "intergenic"

print("CPD SNPs mapped:", len(snp_to_gene_cpd))
n_intergenic = sum(1 for g in snp_to_gene_cpd.values() if g == "intergenic")
print("Intergenic:", n_intergenic, "| With gene:", len(snp_to_gene_cpd) - n_intergenic)

with open(os.path.join(aa_dir, "snp_to_gene_map_cpd_AA.json"), "w") as f:
    json.dump(snp_to_gene_cpd, f)
print("Saved.")

# 3. Load smoker-only genotype subset + CPD phenotype
encoded_df = pd.read_csv(os.path.join(aa_dir, "checkpoint7b_snp_encoded_012_relatedness_filtered.csv"))
meta_df = pd.read_csv(os.path.join(aa_dir, "checkpoint2b_metadata_relatedness_filtered.csv"))
meta_df["sample_id"] = meta_df["sample_id"].astype(str)
meta_df = meta_df.set_index("sample_id")

smoker_ids = meta_df[meta_df["smoking_status"] == "Smoker"].index.tolist()
sample_cols_all = encoded_df.columns[1:].tolist()
smoker_cols = [c for c in sample_cols_all if c in smoker_ids]
print("\nSmoker sample columns found in genotype matrix:", len(smoker_cols))

pheno_cpd = meta_df.loc[smoker_cols, "cpd"].astype(float)
print("CPD phenotype aligned:", pheno_cpd.shape)
print(pheno_cpd.describe())

confounders_cpd = np.load(os.path.join(aa_dir, "confounders_X_cpd.npy"))
print("\nCPD confounders shape:", confounders_cpd.shape, "(should match", len(smoker_cols), "smokers)")

CPD SNPs with resolved positions: 48827
CPD SNPs mapped: 48827
Intergenic: 7559 | With gene: 41268
Saved.

Smoker sample columns found in genotype matrix: 1459
CPD phenotype aligned: (1459,)
count    1459.000000
mean       26.571624
std         6.623071
min         1.000000
25%        20.000000
50%        30.000000
75%        30.000000
max        60.000000
Name: cpd, dtype: float64

CPD confounders shape: (1459, 12) (should match 1459 smokers)


In [6]:
# Check if there's a saved sample-order record from when confounders_X_cpd.npy was built
import os
print([f for f in os.listdir(aa_dir) if 'cpd' in f.lower()])

['checkpoint9_doubleml_stability_results_cpd.csv', 'checkpoint9_doubleml_stability_results_cpd_corrected.csv', 'confounders_X_cpd.npy', 'confounders_X_cpd_1pct.npy', 'gene_map_cpd_corrected.json', 'gene_map_cpd_corrected_grch37.json', 'gene_map_cpd_final_grch37.json', 'pc_col_names_cpd_corrected.json', 'pc_col_names_cpd_final.json', 'pc_input_cpd_corrected.npy', 'pc_input_cpd_final.npy', 'shortlist_cpd_80pct_corrected.csv', 'shortlist_cpd_80pct_final_corrected.csv', 'shortlist_cpd_final.csv', 'snp_to_gene_map_cpd_AA.json', 'sunflower_cpd_AA_final_corrected.png', 'sunflower_v3_cpd_AA_grch37.png']


In [7]:
from sklearn.decomposition import PCA

# Rebuild PCA confounders using smoker_cols order (as we've assumed)
smoker_geno = encoded_df.set_index("probe_id")[smoker_cols].T  # smokers x SNPs
smoker_geno_std = (smoker_geno - smoker_geno.mean(axis=0)) / smoker_geno.std(axis=0)
smoker_geno_std = smoker_geno_std.fillna(0)  # handle any monomorphic-in-subsample SNPs safely

pca = PCA(n_components=5, svd_solver='full')
pcs_rebuilt = pca.fit_transform(smoker_geno_std.values)

print("Rebuilt PC1 vs saved confounders_X_cpd.npy PC1:")
saved_pc1 = confounders_cpd[:, 0]  # assuming col 0 is PC1, adjust if your 12-col layout differs
rebuilt_pc1 = pcs_rebuilt[:, 0]

corr = np.corrcoef(saved_pc1, rebuilt_pc1)[0, 1]
print(f"Correlation: {corr:.4f}  (expect near +1 or -1 if order matches; near 0 if scrambled)")

print("\nAlso checking confounders_X_cpd_1pct.npy for comparison:")
confounders_cpd_1pct = np.load(os.path.join(aa_dir, "confounders_X_cpd_1pct.npy"))
print("Shape:", confounders_cpd_1pct.shape)
if confounders_cpd_1pct.shape[0] == len(smoker_cols):
    corr_1pct = np.corrcoef(confounders_cpd_1pct[:, 0], rebuilt_pc1)[0, 1]
    print(f"Correlation with 1pct version: {corr_1pct:.4f}")

Rebuilt PC1 vs saved confounders_X_cpd.npy PC1:
Correlation: 0.9987  (expect near +1 or -1 if order matches; near 0 if scrambled)

Also checking confounders_X_cpd_1pct.npy for comparison:
Shape: (1631, 12)


In [8]:
# Filter to genic SNPs (CPD-specific mapping)
snp_gene_series_cpd = pd.Series(snp_to_gene_cpd)
snp_gene_series_cpd = snp_gene_series_cpd[snp_gene_series_cpd != "intergenic"]

encoded_df_genic_cpd = encoded_df[encoded_df["probe_id"].isin(snp_gene_series_cpd.index)].copy()
encoded_df_genic_cpd["gene"] = encoded_df_genic_cpd["probe_id"].map(snp_gene_series_cpd)
print("CPD genic SNPs:", len(encoded_df_genic_cpd))

# Keep only smoker columns for CPD analysis
encoded_df_genic_cpd = encoded_df_genic_cpd[["probe_id", "gene"] + smoker_cols]

del encoded_df
gc.collect()

# Compute direction (sign of correlation with CPD, not smoking_status) and flip
geno_matrix_cpd = encoded_df_genic_cpd[smoker_cols].values
pheno_vals_cpd = pheno_cpd.values

geno_centered_cpd = geno_matrix_cpd - geno_matrix_cpd.mean(axis=1, keepdims=True)
pheno_centered_cpd = pheno_vals_cpd - pheno_vals_cpd.mean()

numerator_cpd = geno_centered_cpd @ pheno_centered_cpd
denom_cpd = np.sqrt((geno_centered_cpd**2).sum(axis=1) * (pheno_centered_cpd**2).sum())
denom_cpd[denom_cpd == 0] = np.nan

corr_cpd = numerator_cpd / denom_cpd
direction_cpd = np.sign(np.nan_to_num(corr_cpd, nan=0.0))
direction_cpd[direction_cpd == 0] = 1

print("CPD SNPs flipped:", (direction_cpd < 0).sum())
print("CPD SNPs kept as-is:", (direction_cpd >= 0).sum())

flipped_matrix_cpd = np.where(direction_cpd[:, None] < 0, 2 - geno_matrix_cpd, geno_matrix_cpd)
encoded_df_genic_cpd[smoker_cols] = flipped_matrix_cpd

gene_burden_signed_cpd = encoded_df_genic_cpd.groupby("gene")[smoker_cols].sum()
print("\nAA CPD signed gene burden matrix shape:", gene_burden_signed_cpd.shape)

gene_burden_signed_cpd.to_csv(os.path.join(aa_dir, "gene_burden_matrix_signed_CPD_AA.csv"))

# Filter to protein-coding
gene_burden_cpd_pc = gene_burden_signed_cpd[gene_burden_signed_cpd.index.isin(protein_coding_genes)]
print("AA CPD protein-coding filtered:", gene_burden_cpd_pc.shape)

gene_burden_cpd_pc.to_csv(os.path.join(aa_dir, "gene_burden_matrix_signed_protein_coding_CPD_AA.csv"))
print("Saved final AA CPD gene burden matrix.")

CPD genic SNPs: 41268
CPD SNPs flipped: 20180
CPD SNPs kept as-is: 21088

AA CPD signed gene burden matrix shape: (13162, 1459)
AA CPD protein-coding filtered: (11663, 1459)
Saved final AA CPD gene burden matrix.
